In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

In [13]:
# Cargar el archivo sociodemográfico (escolaridad, edad, sexo)
df_sdem = pd.read_csv('../data/raw/ENOE_SDEMT226.csv', encoding='latin1')

# Cargar el archivo de ocupación y empleo (salarios, horas, ocupación)
df_coe1 = pd.read_csv('../data/raw/ENOE_COE1T226.csv', encoding='latin1')

print("Dimensiones SDEM:", df_sdem.shape)
print("Dimensiones COE1:", df_coe1.shape)

C:\Users\David\AppData\Local\Temp\ipykernel_26328\2843912993.py:2: DtypeWarning: Columns (22: par_c, 23: sex, 24: eda, 25: nac_dia, 26: nac_mes, 27: nac_anio, 28: l_nac_c) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sdem = pd.read_csv('../data/raw/ENOE_SDEMT226.csv', encoding='latin1')
C:\Users\David\AppData\Local\Temp\ipykernel_26328\2843912993.py:5: DtypeWarning: Columns (172: cve_mun) have mixed types. Specify dtype option on import or set low_memory=False.
  df_coe1 = pd.read_csv('../data/raw/ENOE_COE1T226.csv', encoding='latin1')


Dimensiones SDEM: (417128, 115)
Dimensiones COE1: (344709, 174)


In [14]:
df_coe1.head()

,r_def,cd_a,cve_ent,con,upm,d_sem,n_pro_viv,v_sel,n_hog,h_mud,...,p5f14,p5f15,p5f99,ur,tipo,mes_cal,fac_tri,fac_men,cve_mun,cvegeo
0,0,1,9,40002,905953,305,38,3,1,0,...,14,,,1,1,4,895,2760,6,9006
1,0,1,9,40002,905953,305,38,3,1,0,...,,,,1,1,4,895,2760,6,9006
2,0,1,9,40002,905953,305,38,3,1,0,...,,,,1,1,4,895,2760,6,9006
3,0,1,9,40004,906383,311,89,5,1,0,...,14,,,1,1,6,940,3111,6,9006
4,0,1,9,40004,906383,311,89,5,1,0,...,14,,,1,1,6,940,3111,6,9006


In [15]:
df_sdem.head()

,r_def,cve_loc,cve_mun,est,est_d_tri,est_d_men,cve_ageb,t_loc_tri,t_loc_men,cd_a,...,t_tra,emp_ppal,tue_ppal,trans_ppal,mh_fil2,mh_col,sec_ins,tipo,mes_cal,cvegeo
0,0,,7,30,671,671,0,1,1,1,...,0,0,0,0,0,0,0,1,5,9007
1,0,,11,30,671,671,0,1,1,1,...,1,2,2,0,3,2,2,1,6,9011
2,0,,14,40,672,672,0,1,1,1,...,0,0,0,0,0,0,0,1,6,9014
3,0,,2,40,672,672,0,1,1,1,...,1,1,2,0,3,1,2,2,6,9002
4,0,,7,20,670,670,0,1,1,1,...,0,0,0,0,0,0,0,2,6,9007


In [16]:
# Buscar columnas relacionadas con ocupación en el archivo de empleo (coe1)
cols_ocupacion = [col for col in df_coe1.columns if 'ocupa' in col.lower() or 'p3' in col.lower() or 'p4' in col.lower()]
print("Columnas de ocupación encontradas en COE1:", cols_ocupacion)

# Encontrar las columnas que tienen exactamente el mismo nombre en ambos archivos
llaves_comunes = list(set(df_sdem.columns).intersection(set(df_coe1.columns)))
print("Columnas en común disponibles para unir:", llaves_comunes)

Columnas de ocupación encontradas en COE1: ['p3', 'p3a', 'p3b', 'p3c1', 'p3c2', 'p3c3', 'p3c4', 'p3c9', 'p3d', 'p3e', 'p3f1', 'p3f2', 'p3g1_1', 'p3g1_2', 'p3g2_1', 'p3g2_2', 'p3g3_1', 'p3g3_2', 'p3g4_1', 'p3g4_2', 'p3g9', 'p3g_tot', 'p3h', 'p3i', 'p3j1', 'p3j2', 'p3k1', 'p3k2', 'p3k3', 'p3k4', 'p3k5', 'p3k9', 'p3l', 'p4', 'p4_1', 'p4_2', 'p4_3', 'p4a', 'p4a_1', 'p4b', 'p4c', 'p4d1', 'p4d2', 'p4d3', 'p4e', 'p4f', 'p4g', 'p4h', 'p4i', 'p4i_1']
Columnas en común disponibles para unir: ['v_sel', 'upm', 'fac_tri', 'fac_men', 'tipo', 'd_sem', 'n_pro_viv', 'cve_ent', 'cd_a', 'cvegeo', 'con', 'mes_cal', 'r_def', 'h_mud', 'per', 'n_ren', 'n_ent', 'eda', 'cve_mun', 'ur', 'n_hog']


In [17]:
# Verificar cuáles de estas llaves están presentes en ambos DataFrames
llaves_inegi = ['r_def', 'cve_ent', 'con', 'upm', 'd_sem', 'n_pro_v', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

sdem_validas = [k for k in llaves_inegi if k in df_sdem.columns]
coe1_validas = [k for k in llaves_inegi if k in df_coe1.columns]

print("Llaves válidas en SDEM:", sdem_validas)
print("Llaves válidas en COE1:", coe1_validas)

Llaves válidas en SDEM: ['r_def', 'cve_ent', 'con', 'upm', 'd_sem', 'v_sel', 'n_hog', 'h_mud', 'n_ren']
Llaves válidas en COE1: ['r_def', 'cve_ent', 'con', 'upm', 'd_sem', 'v_sel', 'n_hog', 'h_mud', 'n_ren']


In [18]:
# Llaves exactas validadas
llaves = ['r_def', 'cve_ent', 'con', 'upm', 'd_sem', 'v_sel', 'n_hog', 'h_mud', 'n_ren']

# Hacer la unión (merge) final
df_merged = pd.merge(df_sdem, df_coe1, on=llaves, how='inner')

print(f"Dimensiones reales del dataset unificado: {df_merged.shape}")

# Ver los códigos de ocupación más frecuentes para identificar a los docentes
print("\nTop 15 códigos de ocupación:")
print(df_merged['p3'].value_counts().head())

MemoryError: Unable to allocate 2.72 MiB for an array with shape (356854,) and data type object

In [ ]:
# Limpiar espacios en blanco extraños en la columna de ocupación y convertir a texto puro
df_merged['p3'] = df_merged['p3'].astype(str).str.strip()

# Filtrar el DataFrame para quedarnos SOLO con los códigos que empiezan con '23' (Educadores)
df_docentes = df_merged[df_merged['p3'].str.startswith('23', na=False)]

print(f"Total de educadores aislados para el estudio: {df_docentes.shape[0]}")

# Ver qué tipos de profesores tenemos en nuestra muestra
print("\nDesglose de educadores por código SINCO:")
print(df_docentes['p3'].value_counts().head(15))

Total de educadores aislados para el estudio: 6602

Desglose de educadores por código SINCO:
p3
2332    2387
2331    1204
2335     946
2321     892
2322     790
2311     109
2343     101
2312      80
2342      44
2333      19
2399      13
2341      10
2339       5
2334       2
Name: count, dtype: int64


In [ ]:
df_docentes.to_csv('../data/processed/docentes_enoe_optimizado.csv', index=False, encoding='utf-8-sig')
print(f"✅ Dataset exportado: {df_docentes.shape[0]} filas, {df_docentes.shape[1]} columnas")

✅ Dataset exportado: 6602 filas, 280 columnas


# Segunda fase de la limpieza

In [ ]:
df_docentes = pd.read_csv('../data/processed/docentes_enoe_optimizado.csv', na_values=['\\N'])
df_docentes.head()

C:\Users\David\AppData\Local\Temp\ipykernel_26328\4211157128.py:1: DtypeWarning: Columns (22: par_c, 23: sex, 24: eda_x, 25: nac_dia, 26: nac_mes, 27: nac_anio, 28: l_nac_c, 29: cs_p12, 30: cs_p13_1, 31: cs_p13_2, 35: cs_p17, 37: e_con, 38: cs_p20a_1, 39: cs_p20a_c, 40: cs_p20b_1, 41: cs_p20b_c, 208: p4a_1, 209: p4b, 220: p5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_docentes = pd.read_csv('../data/processed/docentes_enoe_optimizado.csv', na_values=['\\N'])


,r_def,cve_loc,cve_mun_x,est,est_d_tri,est_d_men,cve_ageb,t_loc_tri,t_loc_men,cd_a_x,...,p5f14,p5f15,p5f99,ur_y,tipo_y,mes_cal_y,fac_tri_y,fac_men_y,cve_mun_y,cvegeo_y
0,0,,6,40,672,672,0,1,1,1,...,14,,,1,1,5,2044,7421,6,9006
1,0,,120,40,772,772,0,1,1,2,...,14,,,1,1,4,680,2138,120,14120
2,0,,18,40,914,914,0,2,2,3,...,14,,,1,1,6,384,1390,18,19018
3,0,,114,20,961,961,0,1,1,4,...,14,,,1,1,4,143,485,114,21114
4,0,,31,20,706,706,0,2,2,5,...,14,,,1,1,5,122,431,31,11031


In [ ]:
"""def cargar_csv_seguro(ruta: str) -> pd.DataFrame:
    #Carga un CSV manejando problemas de codificación, comillas mal cerradas y filas corruptas.
    codificaciones = ["utf-8", "latin1", "utf-8-sig", "cp1252"]
    separadores = [",", ";", "\t"]

    for encoding in codificaciones:
        for sep in separadores:
            try:
                df = pd.read_csv(ruta, encoding=encoding, sep=sep, on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
    return pd.read_csv(ruta, engine="python", encoding="latin1", quoting=csv.QUOTE_NONE, on_bad_lines="skip")


def limpiar_csv(ruta_entrada: str, ruta_salida: str, reporte_salida: str = None) -> pd.DataFrame:
    if not os.path.exists(ruta_entrada):
        raise FileNotFoundError(f"El archivo '{ruta_entrada}' no existe.")

    df_original = cargar_csv_seguro(ruta_entrada)
    df = df_original.copy()
     
    reporte = []
    reporte.append("==================================================")
    reporte.append("          REPORTE DE LIMPIEZA DE DATOS           ")
    reporte.append("==================================================")
    reporte.append(f"Archivo de origen: {ruta_entrada}")
    reporte.append(f"Filas iniciales: {len(df_original)}")
    reporte.append(f"Columnas iniciales: {len(df_original.columns)}\n")

    # 1. Normalización de nombres de columnas
    cols_originales = list(df.columns)
    df.columns = (
        df.columns.astype(str).str.strip().str.lower()
        .str.replace(" ", "_").str.replace(r"[^\w\s]", "", regex=True)
    )
    cambios_columnas = sum(1 for orig, nuev in zip(cols_originales, df.columns) if orig != nuev)
    reporte.append(f"[1] Columnas renombradas/normalizadas: {cambios_columnas}")

    # 2. Eliminación de filas completamente vacías
    filas_antes = len(df)
    df.dropna(how="all", inplace=True)
    reporte.append(f"[2] Filas totalmente vacías eliminadas: {filas_antes - len(df)}")

    # 3. Eliminación de filas duplicadas (ADAPTADO PARA LLAVE PRIMARIA)
    # Definimos la llave primaria compuesta exacta que se uso en MySQL
    columnas_pk = ['cve_ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_ren']
    
    # Buscamos duplicados basados SOLAMENTE en la llave primaria
    duplicados = df.duplicated(subset=columnas_pk).sum()
    
    # Eliminamos los duplicados. 'keep='last'' conserva el registro más reciente del encuestador
    df.drop_duplicates(subset=columnas_pk, keep='last', inplace=True)
    reporte.append(f"[3] Filas duplicadas (violación de Llave Primaria) eliminadas: {duplicados}")

    # 4. Limpieza de texto en columnas (ADAPTADO PARA INEGI)
    # Convertimos strings vacíos o con solo espacios a nulos matemáticos (NaN)
    reporte.append(f"[4] Estandarización de espacios vacíos a nulos estadísticos (NaN) para bases de datos.")
    df = df.replace(r'^\s*$', np.nan, regex=True)

    # 5. Tratamiento de Valores Faltantes (ADAPTADO PARA MYSQL)
    reporte.append("\n--- Tratamiento de Valores Faltantes (NaN) ---")
    totales_nan = df.isna().sum()
    
    reporte.append("  * NOTA: Por tratarse de datos censales (ENOE) para Machine Learning y MySQL,")
    reporte.append("    se preservan los valores nulos (NaN) para evitar sesgos estadísticos (mediana)")
    reporte.append("    y evitar errores de tipos de datos al exportar a la base de datos relacional.\n")

    for col in df.columns:
        nans = totales_nan[col]
        if nans > 0:
            reporte.append(f"  - Columna '{col}': {nans} valores nulos (Se mantienen como NULL).")

    if totales_nan.sum() == 0:
        reporte.append("  - No se encontraron valores nulos.")

    # Resumen final
    reporte.append("\n==================================================")
    reporte.append("                  RESUMEN FINAL                   ")
    reporte.append("==================================================")
    reporte.append(f"Filas finales: {len(df)}")
    reporte.append(f"Columnas finales: {len(df.columns)}")
    reporte.append(f"Archivo limpio guardado en: {ruta_salida}")

    texto_reporte = "\n".join(reporte)
    print("\n" + texto_reporte + "\n")

    if reporte_salida:
        with open(reporte_salida, "w", encoding="utf-8") as f:
            f.write(texto_reporte)

    # 6. EXPORTACIÓN ADAPTADA PARA MYSQL (na_rep='\\N')
    df.to_csv(ruta_salida, index=False, encoding="utf-8", na_rep='\\N')
    return df

if __name__ == "__main__":
    # Nombres de archivos ajustados para el proyecto final
    archivo_entrada = "../data/processed/docentes_enoe_optimizado.csv"
    archivo_salida = "../data/processed/docentes_enoe_reporte_limpio.csv"
    reporte_txt = "../data/processed/reporte_limpieza.txt"

    df_limpio = limpiar_csv(
        ruta_entrada=archivo_entrada,
        ruta_salida=archivo_salida,
        reporte_salida=reporte_txt,
    )"""

SyntaxError: invalid syntax (908313670.py, line 2)

In [ ]:
def limpiar_y_reportar_docentes(
    ruta_sdemt: str,
    ruta_coe1: str,
    ruta_salida: str = "docentes_enoe_reporte_limpio.csv",
    reporte_salida: str = "reporte_limpieza.txt",
) -> pd.DataFrame:
  reporte = []
  reporte.append("==================================================")
  reporte.append("      REPORTE DE LIMPIEZA - DOCENTES ENOE         ")
  reporte.append("==================================================")

  # 1. Carga de archivos
  if not os.path.exists(ruta_sdemt) or not os.path.exists(ruta_coe1):
    raise FileNotFoundError(
        "No se encontraron los archivos de la ENOE en la ruta especificada."
    )

  reporte.append(f"Archivos de origen: {ruta_sdemt} y {ruta_coe1}")

  # Definir llaves de cruce
  llaves_cruce = ["cve_ent", "con", "upm", "v_sel", "n_hog", "h_mud", "n_ren"]

  # Columnas requeridas
  cols_sdemt = llaves_cruce + [
      "eda",
      "sex",
      "niv_ins",
      "cs_p13_1",  
      "cs_p13_2",
      "anios_esc",
      "hrsocup",
      "ingocup",
      "ing_x_hrs",
      "pos_ocu",
      "seg_soc",
      "clase1",
      "fac_tri",
  ]
  cols_coe1 = llaves_cruce + ["p3", "p3a", "p4", "p4a","p3c1","p3c2","p3c3"]

  df_sdemt = pd.read_csv(
      ruta_sdemt, usecols=cols_sdemt, encoding="latin1", low_memory=False
  )
  df_coe1 = pd.read_csv(
      ruta_coe1, usecols=cols_coe1, encoding="latin1", low_memory=False
  )

  reporte.append(f"Filas iniciales SDEMT: {len(df_sdemt)}")
  reporte.append(f"Filas iniciales COE1: {len(df_coe1)}")

  # 2. Cruce de DataFrames
  df_unido = pd.merge(df_sdemt, df_coe1, on=llaves_cruce, how="inner")
  reporte.append(
      f"[1] Registros totales tras la unión (Inner Join): {len(df_unido)}"
  )

  # 3. Filtrar personal docente (Códigos SINCO del grupo 23)
  df_docentes = df_unido[
      df_unido["p3"].astype(str).str.strip().str.startswith("23")
  ].copy()
  reporte.append(
      f"[2] Filtrado estricto de docentes (Grupo SINCO 23): {len(df_docentes)}"
  )

  # 4. Normalización de nombres de columnas
  cols_originales = list(df_docentes.columns)
  df_docentes.columns = (
      df_docentes.columns.astype(str)
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[^\w\s]", "", regex=True)
  )
  cambios_columnas = sum(
      1 for orig, nuev in zip(cols_originales, df_docentes.columns) if orig != nuev
  )
  reporte.append(f"[3] Columnas normalizadas: {cambios_columnas}")

  # 5. Estandarización de códigos especiales de INEGI ('\\N', espacios vacíos) a NaN
  df_docentes.replace([r"\\N", r"^\s*$", "NA", "NaN"], np.nan, regex=True, inplace=True)

  # 6. Conversión de tipos de datos principales a numéricos
  cols_numericas = ["eda", "sex", "anios_esc", "hrsocup", "ingocup", "fac_tri", "niv_ins", "cs_p13_1"]
  for col in cols_numericas:
    if col in df_docentes.columns:
      df_docentes[col] = pd.to_numeric(df_docentes[col], errors="coerce")

  # 7. Tratamiento inteligente de nulos (Filtrado de estructurales esenciales)
  filas_antes_nulos = len(df_docentes)
  cols_esenciales = ["eda", "sex", "anios_esc", "hrsocup", "ingocup", "fac_tri"]
  df_docentes.dropna(subset=cols_esenciales, inplace=True)
  reporte.append(
      f"[4] Filas eliminadas por valores nulos en variables esenciales"
      f" ({', '.join(cols_esenciales)}): {filas_antes_nulos - len(df_docentes)}"
  )

  # Tratamiento para complementarias (p.ej. contrato p4 -> rellenar nulos con 99 'No especificado')
  if "p4" in df_docentes.columns:
    df_docentes["p4"] = df_docentes["p4"].fillna(99)
    reporte.append(
        "[5] Valores nulos en 'p4' (contrato) rellenados con categoría 99 (No"
        " especificado)."
    )

  # 8. Depuración de valores lógicamente imposibles o absurdos
  filas_antes_atipicos = len(df_docentes)
  df_docentes = df_docentes[
      (df_docentes["eda"] >= 18)
      & (df_docentes["hrsocup"] > 0)
      & (df_docentes["hrsocup"] <= 98)
      & (df_docentes["ingocup"] > 0)
  ].copy()
  reporte.append(
      f"[6] Filas depuradas por valores extraños o fuera de rango (edad < 18,"
      f" horas 0/>98, ingresos <= 0): {filas_antes_atipicos - len(df_docentes)}"
  )

  # 9. Resumen Final
  reporte.append("\n==================================================")
  reporte.append("                  RESUMEN FINAL                   ")
  reporte.append("==================================================")
  reporte.append(f"Filas finales limpias: {len(df_docentes)}")
  reporte.append(f"Columnas finales: {len(df_docentes.columns)}")
  reporte.append(f"Archivo limpio guardado en: {ruta_salida}")

  texto_reporte = "\n".join(reporte)
  print("\n" + texto_reporte + "\n")

  # Guardar reporte en texto
  if reporte_salida:
    with open(reporte_salida, "w", encoding="utf-8") as f:
      f.write(texto_reporte)

  # Guardar CSV final optimizado para MySQL y análisis
  df_docentes.to_csv(ruta_salida, index=False, encoding="utf-8", na_rep="\\N")
  return df_docentes


# --- EJECUCIÓN DESDE EL NOTEBOOK ---
if __name__ == "__main__":
  ruta_sdemt = "data/ENOE_SDEMT226.csv"  # Ajusta tu ruta si es diferente
  ruta_coe1 = "data/ENOE_COE1T226.csv"  # Ajusta tu ruta si es diferente

  df_docentes_limpio = limpiar_y_reportar_docentes(
      ruta_sdemt=ruta_sdemt,
      ruta_coe1=ruta_coe1,
      ruta_salida="docentes_enoe_reporte_limpio.csv",
      reporte_salida="reporte_limpieza.txt",
  )


      REPORTE DE LIMPIEZA - DOCENTES ENOE         
Archivos de origen: data/ENOE_SDEMT226.csv y data/ENOE_COE1T226.csv
Filas iniciales SDEMT: 417128
Filas iniciales COE1: 344709
[1] Registros totales tras la unión (Inner Join): 369419
[2] Filtrado estricto de docentes (Grupo SINCO 23): 6857
[3] Columnas normalizadas: 0
[4] Filas eliminadas por valores nulos en variables esenciales (eda, sex, anios_esc, hrsocup, ingocup, fac_tri): 4
[5] Valores nulos en 'p4' (contrato) rellenados con categoría 99 (No especificado).
[6] Filas depuradas por valores extraños o fuera de rango (edad < 18, horas 0/>98, ingresos <= 0): 3401

                  RESUMEN FINAL                   
Filas finales limpias: 3452
Columnas finales: 27
Archivo limpio guardado en: docentes_enoe_reporte_limpio.csv



C:\Users\fersg\AppData\Local\Temp\ipykernel_42756\3204420791.py:91: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_docentes.dropna(subset=cols_esenciales, inplace=True)
